# Target24 — GRPO Full Experiment

**Запускать после** `check_hard_v2.ipynb` (данные + gold в `data_v2/`)

3 метода:
- **Step 1**: GRPO-only (curriculum + mixing)
- **Step 2**: SFT → GRPO
- **Step 3**: SRFT (SFT warmup + GRPO)

Используется **vLLM** для ускорения генерации при GRPO.

In [ ]:
# Setup (раскомментировать для Colab)
# !pip -q install -U pip
# !pip -q install "transformers>=4.50" "trl>=0.22" datasets accelerate bitsandbytes peft
# !pip -q install unsloth huggingface_hub vllm

In [ ]:
import os, sys
sys.path.insert(0, os.environ.get('PROJECT_ROOT', '.'))
from GRPO_full_experiment import *
print('Imported! USE_VLLM =', USE_VLLM)

## Step 1: GRPO-only (curriculum + mixing)

In [ ]:
model, tokenizer = load_model_lora()
grpo_metrics = train_grpo_curriculum(model, tokenizer, tag='grpo_only', mix_previous=True)
print('GRPO-only training done!')

In [ ]:
# Eval GRPO-only: pass@128 по уровням + диагностика
FastLanguageModel.for_inference(model)
for lvl in range(1, MAX_LEVEL + 1):
    r = eval_pass_at_k(model, tokenizer, lvl, tag='grpo_only')
    pk = r['pass_at_k']
    print(f"L{lvl}: pass@1={pk.get(1,0):.4f}, pass@128={pk.get(128,0):.4f}")

import json
with open(LOGS_DIR / 'grpo_only_pass_at_k.json', 'w') as f:
    json.dump({str(lvl): r['pass_at_k'] for lvl in range(1,MAX_LEVEL+1)}, f, indent=2)

del model, tokenizer
import gc; gc.collect()
import torch
if torch.cuda.is_available(): torch.cuda.empty_cache()

## Step 2: SFT → GRPO

In [ ]:
model, tokenizer = load_model_lora()
print('Phase 1: SFT on gold trajectories')
train_sft(model, tokenizer, tag='sft_phase')
print('Phase 2: GRPO curriculum')
train_grpo_curriculum(model, tokenizer, tag='sft_grpo', mix_previous=True)
print('SFT→GRPO done!')

In [ ]:
FastLanguageModel.for_inference(model)
for lvl in range(1, MAX_LEVEL + 1):
    r = eval_pass_at_k(model, tokenizer, lvl, tag='sft_grpo')
    pk = r['pass_at_k']
    print(f"L{lvl}: pass@1={pk.get(1,0):.4f}, pass@128={pk.get(128,0):.4f}")

with open(LOGS_DIR / 'sft_grpo_pass_at_k.json', 'w') as f:
    json.dump({str(lvl): r['pass_at_k'] for lvl in range(1,MAX_LEVEL+1)}, f, indent=2)

del model, tokenizer; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

## Step 3: SRFT

In [ ]:
model, tokenizer = load_model_lora()
train_srft(model, tokenizer, tag='srft', mix_previous=True)
print('SRFT done!')

In [ ]:
FastLanguageModel.for_inference(model)
for lvl in range(1, MAX_LEVEL + 1):
    r = eval_pass_at_k(model, tokenizer, lvl, tag='srft')
    pk = r['pass_at_k']
    print(f"L{lvl}: pass@1={pk.get(1,0):.4f}, pass@128={pk.get(128,0):.4f}")

with open(LOGS_DIR / 'srft_pass_at_k.json', 'w') as f:
    json.dump({str(lvl): r['pass_at_k'] for lvl in range(1,MAX_LEVEL+1)}, f, indent=2)

del model, tokenizer; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

## Analysis

In [ ]:
run_analysis()

In [ ]:
# Скачать результаты
# import shutil
# shutil.make_archive('experiment_results', 'zip', '.', 'logs_exp')
# from google.colab import files
# files.download('experiment_results.zip')